# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Colorectal Cancer dataset using the `mlcroissant` library via its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant

## 1. Data Loading
Load schema metadata and discover available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and print the top-level dataset metadata (not as a dict, but as .metadata attributes)
print(f"Dataset title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets and their field `@id`s.

In [ ]:
# Find all recordset @ids in the dataset
record_sets = [r['@id'] for r in dataset.metadata._json.get('recordSet', [])]
if not record_sets:
    print("No record sets were declared explicitly in the schema. Attempting fallback: listing record sets via dataset.record_sets property if available.")
    # Try fallback using dataset.record_sets attribute if supported
    try:
        record_sets = [rs['@id'] for rs in dataset.record_sets]
    except Exception:
        record_sets = []

if not record_sets:
    # If no record sets declared, try to infer from schema (for this dataset, likely only one main table)
    # Listing all record sets by iterating over dataset.records() with no arguments
    from itertools import islice
    preview = list(islice(dataset.records(), 1))
    if preview:
        available_keys = list(preview[0].keys())
        print("Sample record keys:", available_keys)
        # We'll treat the whole dataset as a single record set
        default_record_set_id = 'table1'  # Placeholder, as ID is missing. We'll use explicit IDs later when available.
        # Note: In mlcroissant>=0.5, .record_sets may be available, otherwise rely on direct records and doc
    else:
        print("Could not discover any record sets or preview records.")
else:
    print("Record Sets available:")
    for rs_id in record_sets:
        print(f"- {rs_id}")
    # Show field @ids for each record set
    print("\nFields for each Record Set:")
    for rs_id in record_sets:
        # Try to get fields info from the schema JSON
        rs_entry = next((r for r in dataset.metadata._json.get('recordSet', []) if r['@id'] == rs_id), None)
        if rs_entry and 'field' in rs_entry:
            # fields may be a list or a single dict
            fields = rs_entry['field']
            if isinstance(fields, dict):
                fields = [fields]
            field_ids = [f['@id'] for f in fields]
        else:
            field_ids = []
        print(f"  RecordSet {rs_id}: fields: {field_ids}")

## 3. Data Extraction
Load data from the main record set into a DataFrame using the proper record set and field `@id`s discovered above.

In [ ]:
# If actual record sets are not found in the metadata, we load the records directly.
# For this dataset, we expect a single main record set (the patient/sample table).
try:
    # Try with discovered record sets
    assert record_sets
    dataframes = {}
    for record_set_id in record_sets:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    main_rs_id = record_sets[0]
except (AssertionError, Exception):
    # Fallback to loading all records as one DataFrame
    records = list(dataset.records())
    dataframes = {'main': pd.DataFrame(records)}
    main_rs_id = 'main'

print("Columns available:", dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter, normalize, and group data by field `@id`s.

In [ ]:
# Inspect columns, pick numeric and grouping fields
numeric_candidates = dataframes[main_rs_id].select_dtypes(include=['number', 'float', 'int']).columns.tolist()
if not numeric_candidates:
    # Try to coerce columns with likely numeric names
    import numpy as np
    for col in dataframes[main_rs_id].columns:
        try:
            dataframes[main_rs_id][col] = pd.to_numeric(dataframes[main_rs_id][col], errors='ignore')
        except Exception:
            pass
    numeric_candidates = dataframes[main_rs_id].select_dtypes(include=['number', 'float', 'int']).columns.tolist()
# Pick the first numeric column that isn't obviously an ID
numeric_field_id = None
for col in numeric_candidates:
    if 'id' not in col.lower():
        numeric_field_id = col
        break
# Fallback to use the first column if none found
if numeric_field_id is None and numeric_candidates:
    numeric_field_id = numeric_candidates[0]

# For grouping, pick a likely categorical field (not the numeric one)
group_field_candidates = dataframes[main_rs_id].select_dtypes(exclude=['number', 'float', 'int']).columns.tolist()
# Exclude columns with only unique values (eg, IDs)
group_field_id = None
for col in group_field_candidates:
    if col != numeric_field_id and dataframes[main_rs_id][col].nunique() < len(dataframes[main_rs_id]) // 2:
        group_field_id = col
        break

threshold = 10
if numeric_field_id:
    filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by categorical field if possible
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field could be identified for EDA.")

## 5. Visualization
Visualize data distribution for the selected numeric field and grouped aggregate (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution (if numeric_field_id exists)
if numeric_field_id and numeric_field_id in dataframes[main_rs_id]:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), kde=True, color='skyblue')
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

if 'grouped_df' in locals() and group_field_id:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, palette='viridis')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore a FAIR² Croissant dataset using `mlcroissant`. We loaded metadata, explored available record sets and fields (via their `@id`), extracted tabular data, and applied basic filtering, normalization, and group-by operations, all referencing fields by their schema-level `@id`. Visualization illustrated patterns in a numeric field and subgroup means. Next steps could include deeper clinical statistical analysis, model development, or export for domain-specific workflows.